In [ ]:
pip install pandas scipy statsmodels

In [ ]:
import pandas as pd
from scipy.stats import friedmanchisquare
from scipy.stats import wilcoxon
from scipy.stats import chi2
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from itertools import combinations

ALPHA = 0.05

models = [
    "GPT-5.6 Luna",
    "Gemini 3.6 Flash",
    "Qwen 3.7",
    "DeepSeek"
]

print("Significance level (alpha):", ALPHA)
print("\n" + "=" * 60)
print("INTERPRETATION TASK")
print("=" * 60)


file_interpretation = "interpretation_model_evaluation.xlsx"

df_int = pd.read_excel(
    file_interpretation,
    header=1
)

print("\n=== Interpretation Columns ===")
print(df_int.columns.tolist())

print("\n=== Interpretation Shape ===")
print(df_int.shape)

print("\n=== Models ===")
print(df_int["model"].unique())

pivot_f1 = df_int.pivot(
    index="proverb_id",
    columns="model",
    values="BERTScore_F1"
)

pivot_f1 = pivot_f1[models]

pivot_f1 = pivot_f1.dropna()

print("\n=== Data used for Friedman Test ===")
print(pivot_f1.shape)


# FRIEDMAN TEST

statistic, p_value = friedmanchisquare(
    pivot_f1["GPT-5.6 Luna"],
    pivot_f1["Gemini 3.6 Flash"],
    pivot_f1["Qwen 3.7"],
    pivot_f1["DeepSeek"]
)

print("\n=== Friedman Test: Interpretation ===")
print(f"Statistic = {statistic:.4f}")
print(f"p-value   = {p_value:.6f}")
print(f"Alpha     = {ALPHA:.2f}")

if p_value < ALPHA:
    print("Result    = Significant")
else:
    print("Result    = Not significant")


# POST-HOC WILCOXON


if p_value < ALPHA:

    print("\n=== Post-hoc Wilcoxon Signed-Rank Tests ===")

    pair_results = []

    for model_a, model_b in combinations(models, 2):

        statistic_pair, p_value_pair = wilcoxon(
            pivot_f1[model_a],
            pivot_f1[model_b],
            alternative="two-sided"
        )

        pair_results.append({
            "Comparison": f"{model_a} vs {model_b}",
            "Statistic": statistic_pair,
            "p_value": p_value_pair
        })

    pairwise_df = pd.DataFrame(pair_results)

    # HOLM CORRECTION

    reject, p_adjusted, _, _ = multipletests(
        pairwise_df["p_value"],
        alpha=ALPHA,
        method="holm"
    )

    pairwise_df["p_adjusted"] = p_adjusted
    pairwise_df["Significant"] = reject


    # Round values
    pairwise_df["Statistic"] = (
        pairwise_df["Statistic"].round(4)
    )

    pairwise_df["p_value"] = (
        pairwise_df["p_value"].round(6)
    )

    pairwise_df["p_adjusted"] = (
        pairwise_df["p_adjusted"].round(6)
    )


    print(
        pairwise_df.to_string(index=False)
    )

else:

    print("\nFriedman test was not significant.")
    print("No post-hoc Wilcoxon tests were conducted.")

    pairwise_df = pd.DataFrame()


# CONTEXT APPLICATION TASK
# COCHRAN'S Q TEST

print("\n" + "=" * 60)
print("CONTEXT APPLICATION TASK")
print("=" * 60)

file_context = "context_application_results_proverbs_001_100.xlsx"

df_ctx = pd.read_excel(
    file_context,
    header=1
)

print("\n=== Context Application Columns ===")
print(df_ctx.columns.tolist())

print("\n=== Context Application Shape ===")
print(df_ctx.shape)

print("\n=== Models ===")
print(df_ctx["model"].unique())

# CONVERT CORRECT TO BINARY
def convert_correct(value):

    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    if value in ["true", "1", "yes", "correct"]:
        return 1

    if value in ["false", "0", "no", "incorrect"]:
        return 0

    return None


df_ctx["correct_binary"] = (
    df_ctx["correct"].apply(convert_correct)
)

# CREATE PIVOT TABLE

pivot_ctx = df_ctx.pivot(
    index="question_id",
    columns="model",
    values="correct_binary"
)

pivot_ctx = pivot_ctx[models]

# Keep only questions with results from all four models
pivot_ctx = pivot_ctx.dropna()

pivot_ctx = pivot_ctx.astype(int)

print("\n=== Data used for Cochran's Q Test ===")
print(pivot_ctx.shape)


# COCHRAN'S Q TEST

k = pivot_ctx.shape[1]

column_totals = pivot_ctx.sum(axis=0)
row_totals = pivot_ctx.sum(axis=1)

Q = (
    (k - 1)
    *
    (
        k * (column_totals ** 2).sum()
        - column_totals.sum() ** 2
    )
    /
    (
        k * row_totals.sum()
        - (row_totals ** 2).sum()
    )
)

p_value_q = chi2.sf(Q, k - 1)

print("\n=== Cochran's Q Test: Context Application ===")
print(f"Q statistic = {Q:.4f}")
print(f"df          = {k - 1}")
print(f"p-value     = {p_value_q:.6f}")
print(f"Alpha       = {ALPHA:.2f}")

if p_value_q < ALPHA:
    print("Result      = Significant")
else:
    print("Result      = Not significant")


# POST-HOC MCNEMAR
# ONLY IF COCHRAN'S Q TEST IS SIGNIFICANT

if p_value_q < ALPHA:

    print("\n=== Post-hoc McNemar Tests ===")

    context_pair_results = []

    for model_a, model_b in combinations(models, 2):

        # Create 2x2 contingency table
        table = pd.crosstab(
            pivot_ctx[model_a],
            pivot_ctx[model_b]
        )

        # Make sure table is 2x2
        table = table.reindex(
            index=[0, 1],
            columns=[0, 1],
            fill_value=0
        )

        # McNemar test
        result = mcnemar(
            table,
            exact=True
        )

        context_pair_results.append({
            "Comparison": f"{model_a} vs {model_b}",
            "Statistic": result.statistic,
            "p_value": result.pvalue
        })


    # CREATE RESULT TABLE

    context_pairwise_df = pd.DataFrame(
        context_pair_results
    )

    # HOLM CORRECTION

    reject, p_adjusted, _, _ = multipletests(
        context_pairwise_df["p_value"],
        alpha=ALPHA,
        method="holm"
    )

    context_pairwise_df["p_adjusted"] = p_adjusted
    context_pairwise_df["Significant"] = reject


    # Round values
    context_pairwise_df["Statistic"] = (
        context_pairwise_df["Statistic"].round(4)
    )

    context_pairwise_df["p_value"] = (
        context_pairwise_df["p_value"].round(6)
    )

    context_pairwise_df["p_adjusted"] = (
        context_pairwise_df["p_adjusted"].round(6)
    )


    print(
        context_pairwise_df.to_string(index=False)
    )

else:

    print("\nCochran's Q test was not significant.")
    print("No post-hoc McNemar tests were conducted.")

    context_pairwise_df = pd.DataFrame()


# SUMMARY

print("\n" + "=" * 60)
print("STATISTICAL ANALYSIS SUMMARY")
print("=" * 60)

print(f"\nSignificance level: alpha = {ALPHA}")

print("\nInterpretation:")
print(f"  Friedman p-value = {p_value:.6f}")

if p_value < ALPHA:
    print("  Overall result = Significant")
    print("  Post-hoc = Wilcoxon signed-rank + Holm correction")
else:
    print("  Overall result = Not significant")
    print("  Post-hoc = Not conducted")

print("\nContext Application:")
print(f"  Cochran's Q p-value = {p_value_q:.6f}")

if p_value_q < ALPHA:
    print("  Overall result = Significant")
    print("  Post-hoc = Exact McNemar + Holm correction")
else:
    print("  Overall result = Not significant")
    print("  Post-hoc = Not conducted")

Significance level (alpha): 0.05

INTERPRETATION TASK

=== Interpretation Columns ===
['run_id', 'task', 'proverb_id', 'proverb', 'model', 'model_output', 'BERTScore_P', 'BERTScore_R', 'BERTScore_F1']

=== Interpretation Shape ===
(1872, 9)

=== Models ===
['GPT-5.6 Luna' 'Gemini 3.6 Flash' 'Qwen 3.7' 'DeepSeek']

=== Data used for Friedman Test ===
(468, 4)

=== Friedman Test: Interpretation ===
Statistic = 292.4288
p-value   = 0.000000
Alpha     = 0.05
Result    = Significant

=== Post-hoc Wilcoxon Signed-Rank Tests ===
                      Comparison  Statistic  p_value  p_adjusted  Significant
GPT-5.6 Luna vs Gemini 3.6 Flash    28545.0 0.000000    0.000000         True
        GPT-5.6 Luna vs Qwen 3.7    41626.0 0.000006    0.000012         True
        GPT-5.6 Luna vs DeepSeek    21280.0 0.000000    0.000000         True
    Gemini 3.6 Flash vs Qwen 3.7    15811.0 0.000000    0.000000         True
    Gemini 3.6 Flash vs DeepSeek    44404.0 0.000452    0.000452         True
    